# COMP7103 Lab 4 · Audit a pipeline for leakage

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/doubledaibo/COMP7103B/blob/main/labs/lab4/Lab4_leakage_audit.ipynb)

A broadband company wants to know which customers will leave (*churn*). An AI assistant wrote a decision-tree pipeline for the company's customer table, and it reports a test accuracy of about 96%. **Audit the pipeline** for the three forms of leakage from the lecture: target leakage, train/test contamination and duplicates across splits. Fix one leak at a time, and measure the test accuracy before and after each fix.

| Time | What you do |
|---|---|
| 5 min | Brief: run the assistant's pipeline |
| 30 min | Guided work: audit for 3 leakage forms |
| 10 min | Share-out |

This lab is ungraded practice for the exams. Data: synthetic customer table made for this lab (no real customers), CC0.

**Using this notebook in Colab.** Choose *File → Save a copy in Drive* first, and work in your copy. Run a cell with **Shift + Enter**. Run the cells from top to bottom. Where you see `___`, replace it with your own code, then run the cell again.

## 0 · Setup

This cell loads the customer table: one row per exported customer record. The target is `churned` (1 = the customer left).

In [ ]:
import os, warnings, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.base import clone
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectFromModel
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split, ShuffleSplit, GroupShuffleSplit
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
warnings.filterwarnings("ignore")

# A blank (___) that you have not filled in stops with: name '___' is not defined
get_ipython().displayhook.cache_size = 0
globals().pop("___", None)

REPO = "https://raw.githubusercontent.com/doubledaibo/COMP7103B/main/labs/lab4/data/"

def get(name):
    """Use the local file if it is there (Lab 4 downloaded from GitHub), otherwise download it."""
    for path in [name, os.path.join("data", name),]:
        if os.path.exists(path):
            return path
    os.makedirs("data", exist_ok=True)
    try:
        urllib.request.urlretrieve(REPO + name, os.path.join("data", name))
    except Exception:
        raise SystemExit(f"Could not download {name}. Upload it with the folder icon on the left, "
                         "or download the Lab 4 from GitHub: https://github.com/doubledaibo/COMP7103B/tree/main/labs/lab4")
    return os.path.join("data", name)

df = pd.read_csv(get("churn.csv.gz"))
print(f"{len(df):,} rows · {df.shape[1]:,} columns · share who churned: {df['churned'].mean():.2f}")

In [ ]:
df.iloc[:5, :12]   # the first 5 rows and 12 of the columns

## 1 · The assistant's pipeline (5 min)

Run the cell below. It is the code exactly as the assistant wrote it. Read it line by line: you will audit it.

In [ ]:
# ================= as written by the assistant =================
X = df.drop(columns=["customer_id", "churned"])
y = df["churned"]

X = StandardScaler().fit_transform(X)                          # scale every column
select = SelectFromModel(DecisionTreeClassifier(random_state=0),
                         max_features=100, threshold=-np.inf)  # keep the 100 most important columns
X = select.fit_transform(X, y)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0)
model = DecisionTreeClassifier(random_state=0).fit(X_train, y_train)
print(f"Test accuracy: {accuracy_score(y_test, model.predict(X_test)):.1%}")

What the code does:

- It scales every column to mean 0 and standard deviation 1.
- `SelectFromModel` fits a decision tree on all the columns and keeps the 100 columns with the largest *feature importance* (the total drop in Gini impurity from the splits on that column). `threshold=-np.inf` sets no minimum, so exactly 100 columns are kept.
- It splits the rows into 75% training and 25% test, grows a tree on the training rows and scores it on the test rows.

One split can be lucky or unlucky. To compare the versions of the pipeline fairly, we measure each one the same way: the **mean test accuracy over 20 random 75/25 splits** (random subsampling, from the lecture). Run the helper, then measure the assistant's version: pass it the assistant's final `X` and `y`.

In [ ]:
def test_accuracy(model, X, y, groups=None):
    """Mean test accuracy (%) over 20 random 75/25 train/test splits.
    In each split the model is fitted on the training part only and scored on the test part.
    With groups, all rows of one group (one customer) land on the same side of the split."""
    X, y = np.asarray(X), np.asarray(y)
    splitter = (GroupShuffleSplit if groups is not None else ShuffleSplit)(n_splits=20, test_size=0.25, random_state=0)
    accs = [accuracy_score(y[test], clone(model).fit(X[train], y[train]).predict(X[test]))
            for train, test in splitter.split(X, y, groups)]
    return round(100 * float(np.mean(accs)), 1)

results = {}   # version of the pipeline -> mean test accuracy (%)

In [ ]:
results["As written by the assistant"] = test_accuracy(DecisionTreeClassifier(random_state=0), ___, ___)
results

## 2 · Leak 1: target leakage (10 min)

Each row describes one customer on the **snapshot date**, 1 March 2026. The target `churned` says whether the customer left in the 3 months after that date. The model will be used on the snapshot date, so it may only use what is known then. For every column, ask: *when is it recorded?*

| Column | Meaning | When is it recorded? |
|---|---|---|
| `customer_id` | Customer number | At sign-up |
| `tenure_years` | Whole years with the company (6 means 6 or more) | On the snapshot date |
| `contract_months` | Contract length: 1 (month to month), 12 or 24 | On the snapshot date |
| `support_calls_90d` | Calls to customer support in the 90 days before the snapshot | On the snapshot date |
| `fiber_internet` | 1 = fibre internet | On the snapshot date |
| `auto_pay` | 1 = pays by automatic bank debit | On the snapshot date |
| `paperless_billing` | 1 = gets bills by email only | On the snapshot date |
| `senior` | 1 = aged 65 or over | At sign-up |
| `page_0001` … `page_3000` | 1 = opened that help-centre page in the month before the snapshot | On the snapshot date |
| `cancellation_fee_charged` | 1 = the billing system charged a cancellation fee | Billing system |
| `churned` | **The target**: 1 = left in the 3 months after the snapshot | 3 months after the snapshot |

One row names a source instead of a time: decide yourself whether that column is known before or after the snapshot.

Which column is recorded only after the outcome? Write its name, then compare the churn rate for its two values.

In [ ]:
LEAKY = ___       # the column recorded after the outcome, as a string
df.groupby(LEAKY)["churned"].agg(["count", "mean"])

Almost everyone who was charged the fee churned, and almost no one else did: the column is close to a copy of the answer. **Fix 1:** drop it and keep the assistant's other steps unchanged.

In [ ]:
X = df.drop(columns=["customer_id", "churned", ___])
X = StandardScaler().fit_transform(X)       # the assistant's other steps, unchanged
X = select.fit_transform(X, y)
results["Fix 1: drop the leaky column"] = test_accuracy(DecisionTreeClassifier(random_state=0), X, y)
results

## 3 · Leak 2: train/test contamination (10 min)

A step is *fitted* when it learns numbers from the data. The scaler learns each column's mean and standard deviation. The selector fits a tree on `X` and `y` and learns which 100 columns to keep. In the assistant's code both are fitted on all 1,000 rows **before** the split, so the test rows, labels included, helped choose the columns. With 3,000 page columns and only 1,000 rows, some page columns match the churn labels by chance, also on the test rows.

**Fix 2:** put every fitted step in a pipeline. `make_pipeline` chains the steps into one model. `test_accuracy` fits the whole pipeline on the training part of each split, so the test part is never seen. Give it the raw columns, not the already scaled and selected `X`.

In [ ]:
X_raw = df.drop(columns=["customer_id", "churned", LEAKY])     # raw columns: nothing fitted yet
pipe = make_pipeline(___, select, DecisionTreeClassifier(random_state=0))
results["Fix 2: scale and select after the split"] = test_accuracy(pipe, ___, y)
results

Which of the two steps caused the drop? A tree compares one column with a threshold at a time, so scaling a column moves the threshold but not the split. The cell below grows a tree on the 7 customer columns, raw and scaled, and compares the predictions.

In [ ]:
cols = ["tenure_years", "contract_months", "support_calls_90d", "fiber_internet", "auto_pay", "paperless_billing", "senior"]
scaled = StandardScaler().fit_transform(df[cols])
raw_pred = DecisionTreeClassifier(random_state=0).fit(df[cols], y).predict(df[cols])
scaled_pred = DecisionTreeClassifier(random_state=0).fit(scaled, y).predict(scaled)
print(f"Same prediction on {(raw_pred == scaled_pred).mean():.0%} of the rows")

So for a tree, the drop comes from the selection. The scaler before the split is still a leak in principle: for a model that uses distances, such as k-nearest neighbors, it changes the result. It goes in the pipeline too.

## 4 · Leak 3: duplicates across splits (10 min)

Is any customer in the table twice? Count identical rows, and repeated customer numbers.

In [ ]:
print("identical rows:", df.___().sum())
print("repeated customer_id:", df["customer_id"].duplicated().sum())

How many test rows have a copy in the training rows? The cell below makes the assistant's split again (same `test_size` and `random_state`, so the same rows) on the customer numbers.

In [ ]:
ids_train, ids_test = train_test_split(df["customer_id"], test_size=0.25, random_state=0)   # the assistant's split
print(f"{ids_test.isin(ids_train).mean():.0%} of the test rows have a copy in the training rows")

A fully grown tree puts a training row and its copy in the same leaf, so each copy is a free correct answer. **Fix 3:** remove the duplicates *before* the split, so each customer is in the table once.

In [ ]:
clean = df.___()                                     # one row per customer
X_clean = clean.drop(columns=["customer_id", "churned", LEAKY])
results["Fix 3: remove duplicates before the split"] = test_accuracy(pipe, X_clean, clean["churned"])
results

Another fix keeps all rows but **splits by customer**: all rows of one customer go to the same side (`GroupShuffleSplit` in the helper). It should give about the same accuracy.

In [ ]:
print("split by customer:", test_accuracy(pipe, X_raw, y, groups=df["customer_id"]), "%")

**Before and after.** Each fix is measured on top of the fixes before it. The dashed line is the majority baseline (ZeroR in the lecture): always predict the larger class (the customer stays). The purple bar is the honest estimate after all three fixes.

In [ ]:
table = pd.DataFrame({"test accuracy (%)": results})
table["change (points)"] = table["test accuracy (%)"].diff().map(lambda v: "" if pd.isna(v) else f"{v:+.1f}")
baseline = 100 * max(clean["churned"].mean(), 1 - clean["churned"].mean())

fig, ax = plt.subplots(figsize=(8, 3))
ax.barh(table.index[::-1], table["test accuracy (%)"][::-1], color=["#6d4fc2"] + ["#bbbbbb"] * (len(table) - 1))
ax.axvline(baseline, ls="--", color="k", lw=1)
ax.text(baseline + 0.5, len(table) - 0.45, f"majority baseline {baseline:.0f}%", fontsize=8)
for i, v in enumerate(table["test accuracy (%)"][::-1]):
    ax.text(v + 0.5, i, f"{v:.1f}%", va="center", fontsize=9)
ax.set_xlim(0, 105)
ax.set_ylim(-0.6, len(table) - 0.1)
ax.set_xlabel("mean test accuracy over 20 splits (%)")
ax.set_title("The same tree, before and after each leak is fixed")
plt.show()
table

**Your audit.** For each leakage form: was it in the pipeline (found / not found)? Quote the line of code.

In [ ]:
AUDIT = {
    "1 · Target leakage": "found / not found, and the line of code",
    "2 · Train/test contamination": "found / not found, and the line of code",
    "3 · Duplicates across splits": "found / not found, and the line of code",
}
for form, verdict in AUDIT.items():
    print(form, "·", verdict)

## 5 · Share-out (10 min)

- For each of the three leakage forms, give your verdict (found / not found) and the line of code; we vote on the class board.
- Report the mean test accuracy from your table, not the single split the assistant printed: **before the fix** = *As written by the assistant*, **after the fix** = *Fix 3* (all three fixes), in %; we enter the class's numbers on the board.
- Which fix changed the accuracy most? Which leak would you have missed without the data dictionary?
- The habit from the lecture: every column is checked for *when it is recorded*, every fitted step goes after the split, and duplicates are removed (or the split is made by customer) before anything else.